# Bradford 0-19 Children and Young Peoples' Outcomes Framework: Classification metrics for ASQ GLD predicting EYFSP GLD

The purpose of this notebook is to calculate classification metrics for the relationship between ASQ GLD status and EYFSP GLD status.

Specifically, this notebook examines whether ASQ GLD status at the 2-2½-year review can identify children who later achieve a Good Level of Development (GLD) in the EYFSP.

Two types of classification metrics are considered:

1. **Descriptive classification metrics based on observed ASQ and EYFSP GLD status**  

   In this approach, ASQ GLD status is treated as a simple binary classifier of EYFSP GLD status. The metrics are calculated directly from the observed ASQ GLD and EYFSP GLD variables, without using model-predicted probabilities.

2. **Apparent classification metrics based on regression model predictions**  

   In this approach, logistic regression models are used to estimate each child's predicted probability of achieving EYFSP GLD. These predicted probabilities are then used to calculate discrimination and classification metrics, including AUC, sensitivity and specificity at selected probability thresholds.

The main metrics reported are:

- confusion matrix
- sensitivity
- specificity
- accuracy
- AUC

Because the regression models are fitted and evaluated in the same analytic sample, the model-based metrics should be interpreted as **apparent performance**, rather than out-of-sample predictive performance.

# Preliminary steps: load libraries, set up database connection, and load helper functions

In [ ]:
import os
import contextlib

with open(os.devnull, "w") as f, contextlib.redirect_stderr(f):
    import rpy2.robjects as ro

%load_ext rpy2.ipython

In [ ]:
import pandas as pd
from rich import print
from IPython.display import display, Image
from sqlalchemy import create_engine
from importlib import reload
from pathlib import Path

import plotly.io as pio
pio.renderers.default = "notebook_connected"

from utils import (
    show_tab_model,
    make_crosstab
)

import warnings
warnings.filterwarnings('ignore')

## Load data

In [ ]:
#| echo: false

tbl_name = 'person_linked_2016plus'

# Note: The connection string is specific to the Connected Bradford VDE.
# Replace the placeholder below with the internal server URL.
conn_str = "DATABASE_URL_PLACEHOLDER"

# Create SQLAlchemy engine
engine = create_engine(conn_str)

df = pd.read_sql(f"SELECT * FROM [dbo].[{tbl_name}_derived]", engine)

print(df.shape)

## Load R helper functions

In [ ]:
%%R -i use_mock_set

source("R/setup.R")
source("R/prepare_data.R")
source("R/model_tables.R")
source("R/classification_metrics.R")
source("R/save_outputs.R")

## Create person table

In [ ]:
df.drop(columns=["FSP_GLD"], inplace=True)

# rename columns
df.rename(columns={
    'Valid_2y_HV': "Has_HV",
    "Valid_2y_ASQ": "Has_ASQ",
    "ASQ_DomainBinary_FGLD": "ASQ_FGLD_dom",
    "FSP_GLD_derived": "FSP_GLD",
    "ASQ_PHE_Risk": "ASQ_GLD"
}, inplace=True)


df['FSP_Present'] = df['FSP_Present'].map({True: 1, False: 0})

In [ ]:
p_cols = ['ethnicity_group', 'gender', 'IMD19_deciles', 'IMD19_quintile', 'age_2_5', 'gender_raw', 'birth_datetime', 'FSP_ACADYR']

derived_cols = ['Has_HV', 'Has_ASQ', 'FSP_GLD', 'FSP_TotalScore', 'ASQ_FGLD', 'ASQ_GLD', 'ASQ_Composite', 'age_fsp_months', 'FSP_Present', "ASQ_Status", 'ASQ_Version', 'ASQ_n_domains']

df_person = df[['person_id'] + p_cols + derived_cols].drop_duplicates()

df_person = (
    df_person
    .sort_values(
        by=['Has_ASQ', 'ASQ_Version'],
        ascending=[False, False]   # priority: Yes + latest version
    )
    .drop_duplicates(subset='person_id', keep='first')
)

assert df_person['person_id'].is_unique, "person_id is not unique in df_person"

df_person_domains = pd.merge(
    df_person,
    df[['person_id', 'ASQ_Version', 'ASQ_Domain', 'ASQ_FGLD_dom', 'Has_ASQ'] + [c for c in df.columns if c.startswith('FSP_') and '_Binary' in c]].drop_duplicates(),
    on=['person_id', 'ASQ_Version', 'Has_ASQ'],
    how='left'
)

def check_conflict(df, col):
    bad = df.groupby("person_id")[col].nunique(dropna=True)
    conflict_ids = bad[bad > 1]
    if len(conflict_ids) > 0:
        print(f"⚠️ {col} has {len(conflict_ids)} conflicting persons")
        return df[df.person_id.isin(conflict_ids.index)][['person_id', col]].sort_values('person_id')
    # else:
    #     print(f"✓ {col} OK")

for col in p_cols + derived_cols:
    check_conflict(df_person, col)

In [ ]:
df_fsp_asq = df_person_domains[(df_person_domains['FSP_GLD'].notna()) & (df_person_domains['ASQ_GLD'].notna())].drop_duplicates()
df_person_fsp_asq = df_fsp_asq[['person_id', 'gender', 'ethnicity_group', 'ASQ_Composite', 'ASQ_FGLD', 'ASQ_GLD', 'FSP_TotalScore', 'FSP_GLD', 'age_fsp_months', 'IMD19_deciles', 'IMD19_quintile']].drop_duplicates()

print(f"[bold cyan]Eligible sample: {df_person_fsp_asq.person_id.nunique():,} unique children who had a non-empty FSP_GLD and complete ASQ entries.[/bold cyan]")

# Descriptive classification performance: ASQ GLD vs EYFSP GLD

The metrics below are calculated directly from the cross-tabulation of observed `ASQ_GLD` and observed `FSP_GLD`.

In this section, `ASQ_GLD` is treated as a binary early indicator of later EYFSP GLD status, **the positive class is defined as `FSP_GLD = 1`**:

- `ASQ_GLD = 1`: the child achieved GLD according to the ASQ-based measure.
- `ASQ_GLD = 0`: the child did not achieve GLD according to the ASQ-based measure.
- `FSP_GLD = 1`: the child later achieved GLD in the EYFSP.
- `FSP_GLD = 0`: the child later did not achieve GLD in the EYFSP.


The confusion matrix is defined as:

|  | Observed `FSP_GLD = 1` | Observed `FSP_GLD = 0` |
|---|---:|---:|
| `ASQ_GLD = 1` | True positive (TP) | False positive (FP) |
| `ASQ_GLD = 0` | False negative (FN) | True negative (TN) |

In this context:

- **True positive (TP)**: the child achieved ASQ GLD and later achieved EYFSP GLD.
- **False positive (FP)**: the child achieved ASQ GLD but later did not achieve EYFSP GLD.
- **False negative (FN)**: the child did not achieve ASQ GLD but later achieved EYFSP GLD.
- **True negative (TN)**: the child did not achieve ASQ GLD and later did not achieve EYFSP GLD.

The classification metrics are then calculated as:

- **Sensitivity** = `TP / (TP + FN)`  
  Among children who later achieved EYFSP GLD, this is the proportion who had achieved ASQ GLD.

- **Specificity** = `TN / (TN + FP)`  
  Among children who later did not achieve EYFSP GLD, this is the proportion who had not achieved ASQ GLD.

- **Positive predictive value (PPV)** = `TP / (TP + FP)`  
  Among children who achieved ASQ GLD, this is the proportion who later achieved EYFSP GLD.

- **Negative predictive value (NPV)** = `TN / (TN + FN)`  
  Among children who did not achieve ASQ GLD, this is the proportion who later did not achieve EYFSP GLD.

- **Accuracy** = `(TP + TN) / (TP + FP + FN + TN)`  
  This is the overall proportion of children whose ASQ GLD status matched their later EYFSP GLD status.

- **AUC** = `(Sensitivity + Specificity) / 2`  
  Because `ASQ_GLD` is binary, the AUC is equivalent to the average of sensitivity and specificity. It summarises how well the binary ASQ GLD indicator distinguishes between children who later achieved EYFSP GLD and those who did not.

Because `ASQ_GLD` is a binary observed indicator rather than a continuous score or model-predicted probability, the AUC should be interpreted as a summary of the classification performance of this binary indicator. It is not based on a fitted prediction model.

In [ ]:
df_crosstab = make_crosstab(
    df_person_fsp_asq,
    row_var="ASQ_GLD",
    col_var='FSP_GLD',
    caption_prefix="FSP_GLD x ASQ_GLD"
)

In [ ]:
ct = df_crosstab.copy()

# extract count from cells like "1540 (57.98%)"
def get_count(x):
    return int(re.match(r"^\s*([0-9,]+)", str(x)).group(1).replace(",", ""))

fsp0_col = [c for c in ct.columns if "FSP_GLD" in str(c) and "0" in str(c)][0]
fsp1_col = [c for c in ct.columns if "FSP_GLD" in str(c) and "1" in str(c)][0]

tn = get_count(ct.loc[0, fsp0_col])
fn = get_count(ct.loc[0, fsp1_col])
fp = get_count(ct.loc[1, fsp0_col])
tp = get_count(ct.loc[1, fsp1_col])

# Calculate metrics
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)
npv = tn / (tn + fn)
accuracy = (tp + tn) / (tp + tn + fp + fn)
auc = (sensitivity + specificity) / 2

metrics_from_crosstab = pd.DataFrame({
    "Metric": [
        "Sensitivity",
        "Specificity",
        "Positive predictive value",
        "Negative predictive value",
        "Accuracy",
        "AUC"
    ],
    "Value": [
        sensitivity,
        specificity,
        ppv,
        npv,
        accuracy,
        auc
    ]
})

metrics_from_crosstab["Value"] = metrics_from_crosstab["Value"].round(3)

metrics_from_crosstab

# Apparent classification performance: ASQ GLD vs EYFSP GLD

The metrics below are calculated from logistic regression model predictions.

In this section, `ASQ_GLD` is used as a predictor in the regression model, together with any other covariates included in the model. The outcome is `FSP_GLD`, where:

- `FSP_GLD = 1`: the child later achieved GLD in the EYFSP.
- `FSP_GLD = 0`: the child later did not achieve GLD in the EYFSP.

The logistic regression model estimates each child's predicted probability of achieving EYFSP GLD:

`Predicted probability = P(FSP_GLD = 1 | predictors)`

To calculate threshold-based classification metrics, the predicted probabilities are converted into predicted classes using a selected threshold. For example, if the threshold is 0.5:

- predicted probability ≥ 0.5 → predicted `FSP_GLD = 1`
- predicted probability < 0.5 → predicted `FSP_GLD = 0`

The confusion matrix is then defined as:

|  | Observed `FSP_GLD = 1` | Observed `FSP_GLD = 0` |
|---|---:|---:|
| Predicted `FSP_GLD = 1` | True positive (TP) | False positive (FP) |
| Predicted `FSP_GLD = 0` | False negative (FN) | True negative (TN) |

In this context:

- **True positive (TP)**: the model predicted that the child would achieve EYFSP GLD, and the child did achieve EYFSP GLD.
- **False positive (FP)**: the model predicted that the child would achieve EYFSP GLD, but the child did not achieve EYFSP GLD.
- **False negative (FN)**: the model predicted that the child would not achieve EYFSP GLD, but the child did achieve EYFSP GLD.
- **True negative (TN)**: the model predicted that the child would not achieve EYFSP GLD, and the child did not achieve EYFSP GLD.

The classification metrics are then calculated as:

- **Sensitivity** = `TP / (TP + FN)`  
  Among children who later achieved EYFSP GLD, this is the proportion correctly classified by the model as predicted to achieve EYFSP GLD.

- **Specificity** = `TN / (TN + FP)`  
  Among children who later did not achieve EYFSP GLD, this is the proportion correctly classified by the model as predicted not to achieve EYFSP GLD.

- **Positive predictive value (PPV)** = `TP / (TP + FP)`  
  Among children classified by the model as predicted to achieve EYFSP GLD, this is the proportion who actually achieved EYFSP GLD.

- **Negative predictive value (NPV)** = `TN / (TN + FN)`  
  Among children classified by the model as predicted not to achieve EYFSP GLD, this is the proportion who actually did not achieve EYFSP GLD.

- **Accuracy** = `(TP + TN) / (TP + FP + FN + TN)`  
  This is the overall proportion of children whose model-predicted EYFSP GLD class matched their observed EYFSP GLD status.

- **AUC**  
  AUC is calculated from the model-predicted probabilities, before applying a classification threshold. It summarises how well the model ranks children who achieved EYFSP GLD above children who did not achieve EYFSP GLD.

AUC is calculated directly from the predicted probabilities and does not require a classification threshold. However, sensitivity, specificity, PPV, NPV and accuracy require the predicted probabilities to be converted into predicted classes using a selected threshold, such as 0.5.

These metrics describe the model's performance within the current sample and **should not be interpreted as out-of-sample predictive performance or external validation**.

In [ ]:
df_tmp = df_person_fsp_asq.copy()

In [ ]:
%%R -i df_tmp -o output_file_html

same_name <- "metrics_reg_model"
out <- make_outputs(results_dir, same_name)
output_file_html <- out$html

df_tmp <- prepare_df(df_tmp, numeric_cols=c("age_fsp_months", "ASQ_Composite"))

model_glm1 <- glm(
  FSP_GLD ~ ASQ_GLD,
  data = df_tmp,
  family = "binomial"
)

model_glm2 <- glm(
  FSP_GLD ~ ASQ_GLD + gender + ethnicity + age_months + IMD,
  data = df_tmp,
  family = "binomial"
)

models_rq52 <- list(model_glm1, model_glm2)
saveRDS(models_rq52, file = out$rds)

dv_labels <- c(
  "EYFSP: Odds of achieving GLD by ASQ-3 GLD status (unadjusted)",
  "EYFSP: Odds of achieving GLD by ASQ-3 GLD status (adjusted)"
)

dv_labels <- paste0("**", dv_labels, "**")

tbl_list <- list()

for (i in seq_along(models_rq52)) {
  tbl_list[[i]] <- make_reg_tbl(models_rq52[[i]])
}

final_table <- make_merged_table(models_rq52, dv_labels, sort_predictors = TRUE)
  
gt_final_table <- make_gt(final_table)

gt::gtsave(gt_final_table, filename = output_file_html)
saveRDS(gt_final_table, file = out$rds_gt)
save_docs && safe_docx(gt_final_table, filename = out$docx)

# summary(model_glm1)

In [ ]:
p = Path(output_file_html[0])
print(f"[green]Table saved to:[/] {p.parent.name}/{p.name}")

show_tab_model(output_file_html)
del df_tmp

In [ ]:
%%R -o combined_html

model_names <- c(
  "ASQ-3 GLD status (unadjusted)",
  "ASQ-3 GLD status (adjusted)"
)

class_outputs <- purrr::map(
  models_rq52,
  ~ get_classification(.x, outcome = "FSP_GLD", threshold = 0.5)
)

# Classification metrics table
class_tbl <- purrr::map2_dfr(
  class_outputs,
  model_names,
  ~ .x$metrics %>%
    dplyr::mutate(Model = .y, .before = 1)
)

class_gt <- class_tbl %>%
  gt::gt() %>%
  gt::cols_move_to_start(columns = Model) %>%
  gt::fmt_number(
    columns = c(
      Sensitivity,
      Specificity,
      Accuracy,
      AUC,
      `Positive predictive value`,
      `Negative predictive value`
    ),
    decimals = 3
  ) %>%
  gt::tab_header(
    title = "Classification performance for EYFSP GLD prediction"
  )


# Confusion matrix table
cm_tbl <- purrr::map2_dfr(
  class_outputs,
  model_names,
  ~ .x$confusion_matrix %>%
    dplyr::mutate(Model = .y, .before = 1)
) %>%
  dplyr::select(
    Model,
    Predicted,
    `Actual GLD = 0`,
    `Actual GLD = 1`
  ) %>%
  dplyr::group_by(Model) %>%
  dplyr::mutate(
    Model_display = dplyr::if_else(dplyr::row_number() == 1, Model, "")
  ) %>%
  dplyr::ungroup() %>%
  dplyr::select(
    Model = Model_display,
    Predicted,
    `Actual GLD = 0`,
    `Actual GLD = 1`
  )

cm_gt <- cm_tbl %>%
  gt::gt() %>%
  gt::sub_missing(
    columns = Model,
    missing_text = ""
  ) %>%
  gt::tab_header(
    title = "Confusion matrices for EYFSP GLD prediction"
  ) %>%
  gt::tab_source_note(
    gt::md(
      "Rows are predicted GLD status; columns are actual GLD status. Positive class is GLD = 1."
    )
  )

cm_gt <- cm_tbl %>%
  gt::gt() %>%
  gt::cols_move_to_start(columns = Model) %>%
  gt::tab_header(
    title = "Confusion matrices for EYFSP GLD prediction"
  ) %>%
  gt::tab_source_note(
    gt::md(
      "Rows are predicted GLD status; columns are actual GLD status. Positive class is GLD = 1."
    )
  )


# Combine both tables into one HTML
combined_html <- file.path(
  results_dir,
  paste0(same_name, "_classification_and_confusion_fig.html")
)

combined_page <- htmltools::tagList(
  htmltools::tags$div(gt::as_raw_html(cm_gt)),
  htmltools::tags$br(),
  htmltools::tags$hr(),
  htmltools::tags$br(),
  htmltools::tags$div(gt::as_raw_html(class_gt))
)

htmltools::save_html(combined_page, file = combined_html)

In [ ]:
p = Path(combined_html[0])
print(f"[green]Table saved to:[/] {p.parent.name}/{p.name}")

show_tab_model(combined_html)